### Procesamiento de Lenguaje Natural I
# **Desafío 1**



### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [1]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [2]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [3]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [4]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [5]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [6]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [7]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [8]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [9]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [10]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [11]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [12]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [13]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [14]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [15]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [16]:
np.argsort(cossim)[::-1]

array([4811, 6635, 4253, ..., 9019, 9016, 8748], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [17]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [18]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [19]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [20]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](20,)","[480.,584.,591.,...,564.,465.,377.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](20,)","[-3.16,-2.96,-2.95,...,-3. ,-3.19,-3.4 ]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](20,)","[ 0, 1, 2,...,17,18,19]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](20, 101631)","[[0. ,0.94,0. ,...,0. ,0. ,0. ], [1.39,0.6 ,0. ,...,0. ,0. ,0. ], [0.95,0.14,0. ,...,0. ,0. ,0. ], ..., [0.42,2.9 ,0.04,...,0. ,0. ,0. ], [0.61,1.36,0. ,...,0. ,0. ,0. ], [0.03,0.38,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](20, 101631)","[[-11.56,-10.9 ,-11.56,...,-11.56,-11.56,-11.56], [-10.69,-11.1 ,-11.56,...,-11.56,-11.56,-11.56], [-10.9 ,-11.44,-11.57,...,-11.57,-11.57,-11.57], ..., [-11.22,-10.21,-11.54,...,-11.57,-11.57,-11.57], [-11.09,-10.7 ,-11.56,...,-11.56,-11.56,-11.56], [-11.53,-11.23,-11.56,...,-11.56,-11.56,-11.56]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,101631


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [21]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [22]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


---------------

### Consigna 1

In [ ]:
# Muestreo 5 indices
random_idxs = np.random.choice(len(newsgroups_train.data) + 1, size=5, replace=False)
for idx in random_idxs:
    print('='*60) 
    print(f'indice: {idx}')
    print(f'clase: {newsgroups_train.target_names[newsgroups_train.target[idx]]}')
    print('-'*60)
    print(newsgroups_train.data[idx])

indice: 8303
clase: sci.electronics
------------------------------------------------------------
 So-Called Cool-Hot boxes have been advetised for several years.  I recall
Damark advertising them in a recent catalog.  Problem with the units is they do
a sh***y job of keeping food cold/warm.  The peliter devices used just don't
seem to have enough punch to keep up.  If you want something hot, you need to
heat it up before you put it in the box, and end up hours later with food thats
only moderately warm.  Same goes for cold items.


 You'de probably be better off getting a good Coleman(tm) cooler and stocking
up on "blue ice" blocks.

 Same as above.  Not enough "punch" in them to keep/get things cold/hot.
Mike Behnke             | Senior Tech/Advisor | Quid est illuidin aqua??
Fermi Nat Accel Lab     | Equipment Suuport   |
Batavia, Il.            | Computing Div       | PISTRIX!!  PISTRIX!!
BEHNKE@FNALF.FNAL.GOV   |                     |

My opinions are my own, not of the lab. So, if

In [39]:
for idx in random_idxs:
  print(f'Clase original: {newsgroups_train.target_names[newsgroups_train.target[idx]]}')
  cossim = cosine_similarity(X_train[idx], X_train)[0]
  mostsim = np.argsort(cossim)[::-1][1:6]
  print('Clases de los 5 documentos mas similares:')
  for i in mostsim:
    print(newsgroups_train.target_names[y_train[i]])
  print('='*60)

Clase original: sci.electronics
Clases de los 5 documentos mas similares:
sci.electronics
talk.politics.guns
talk.politics.guns
sci.med
rec.sport.hockey
Clase original: misc.forsale
Clases de los 5 documentos mas similares:
misc.forsale
misc.forsale
misc.forsale
misc.forsale
comp.sys.ibm.pc.hardware
Clase original: sci.med
Clases de los 5 documentos mas similares:
sci.med
sci.med
soc.religion.christian
sci.med
talk.religion.misc
Clase original: sci.space
Clases de los 5 documentos mas similares:
sci.space
sci.space
talk.religion.misc
sci.space
sci.space
Clase original: sci.med
Clases de los 5 documentos mas similares:
sci.med
sci.med
sci.med
sci.med
sci.med


#### Analisis de resultados

##### Documento 1 (idx 8303)
* **Clase original:** `sci.electronics`
* **Vecinos mas cercanos:** `sci.electronics`, `talk.politics.guns`, `talk.politics.guns`, `sci.med`, `rec.sport.hockey`
* **Analisis:**
  El texto trata sobre un dispositivo electronico para mantener comida fría o caliente y lo deficiente de su funcionamiento. Solo la primera coincidencia pertenece a la clase correcta. El vocabulario utilizado no es del todo adjudicable a la categoria e inclusive la categoria asignada no es del todo inferible asi que es esperable la dispersion.

---

##### Documento 2 (idx 9663)
* **Clase original:** `misc.forsale`
* **Vecinos mas cercanos:** `misc.forsale`, `misc.forsale`, `misc.forsale`, `misc.forsale`, `comp.sys.ibm.pc.hardware`
* **Analisis:**
  El resultado es bueno, 4 de los 5 documentos pertenecen exactamente a `misc.forsale`, y el quinto pertenece a hardware de PC. El texto utiliza frases y términos fuertemente asociados a la compraventa ("for sale", "brand new in the box", "never used", "asking $40.00"). Ademas, el producto específico es una impresora, por lo que la aparición de `comp.sys.ibm.pc.hardware` tiene sentido semántico.

---

##### Documento 3 (idx 5188)
* **Clase original:** `sci.med`
* **Vecinos mas cercanos:** `sci.med`, `sci.med`, `soc.religion.christian`, `sci.med`, `talk.religion.misc`
* **Analisis:**
  El documento es una consulta médica sobre un electrocardiograma ("EKG"), latidos irregulares y niveles bajos de potasio en sangre. La mayoría (3 de 5) acierta en `sci.med` gracias a términos médicos como EKG, potassium y doctor. Sin embargo, aparecen dos documentos vinculados a religión (`soc.religion.christian` y `talk.religion.misc`) cuya vinculacion no es del todo explicable.

---

##### Documento 4 (idx 7789)
* **Clase original:** `sci.space`
* **Vecinos mas cercanos:** `sci.space`, `sci.space`, `talk.religion.misc`, `sci.space`, `sci.space`
* **Analisis:**
  El resultado es muy bueno, 4 de los 5 documentos pertenecen a `sci.space`. El contenido describe la órbita de Mercurio, relatividad, física newtoniana, el descubrimiento de planetas (Neptuno, Urano) y asteroides. EL buen resultado pueude ser adjudicable al uso de vocabulario técnico y astronómico warped space, Mercury, Newtonian, Neptune, Uranus, planet, asteroid, sun. La aparicion de `talk.religion.misc` puede explicarse por el nombre de los dioses de algunas religiones, que son similares a los observados en este texto.

---

##### Documento 5 (idx 3416)
* **Clase original:** `sci.med`
* **Vecinos mas cercanos:** `sci.med`, `sci.med`, `sci.med`, `sci.med`, `sci.med`
* **Analisis:**
  Se obtuvo una coincidencia perfecta. El texto discute prescripciones de medicaciones "Codeine", "prescribe", "Tylenol #3", "refills", "acetominophen". Estas palabras al ser tan especificas de la categoria son las causales del gran resultado obtenido.




### Consigna 2

In [49]:
#Anteriormente X_test y y_test fueron vectorizados asi qeu se usan directamente
from sklearn.metrics import classification_report

sim_matrix = cosine_similarity(X_test, X_train)
best_train_indices = np.argmax(sim_matrix, axis=1)

# Mapeamos indices
y_pred = y_train[best_train_indices]

print(classification_report(
    y_test, 
    y_pred, 
    target_names=newsgroups_train.target_names,
))

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.57       396
         sci.electronics       0.53      0.33      0.41       393
                 sci.med       0.65      0.49      0.56       396
         

#### Discusion

Este modelo simple que consite en comparar a fuerza bruta un documento con todo el conjunto de entrenamiento nos permite obtener un baseline para futuras comparaciones. El F1 Macro obtenido fue de 0.5 y en general no tiene buen desmepeno en ninguna clase, excepto `rec.sport.hockey`

### Consigna 3

In [23]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, classification_report
import pandas as pd

X_train_docs = newsgroups_train.data
y_train = newsgroups_train.target

X_test_docs = newsgroups_test.data
y_test = newsgroups_test.target


In [25]:
# Para probrar varios parametros hacemos busqueda en grilla
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english')),
    ('clf', MultinomialNB())
])

param_grid = [
    # Experimentos para MultinomialNB
    {
        'clf': [MultinomialNB()],
        'clf__alpha': [1.0, 0.5, 0.1, 0.05, 0.01],
        'tfidf__sublinear_tf': [True, False],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.5, 0.8, 1.0],
    },
    # Experimentos para ComplementNB
    {
        'clf': [ComplementNB()],
        'clf__alpha': [1.0, 0.5, 0.1, 0.05, 0.01],
        'clf__norm': [True, False],
        'tfidf__sublinear_tf': [True, False],
        'tfidf__min_df': [1, 2, 3],
        'tfidf__max_df': [0.5, 0.8, 1.0],
    }
]

# Hacemos la busqueda con CV de 5
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='f1_macro',
    cv=5,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_docs, y_train)

print(f"\nMejor F1-Macro en val: {grid_search.best_score_:.4f}")
print("Mejores parametros:")
for param, val in grid_search.best_params_.items():
    print(f"  {param}: {val}")


Fitting 5 folds for each of 270 candidates, totalling 1350 fits



Mejor F1-Macro en val: 0.7628
Mejores parametros:
  clf: ComplementNB()
  clf__alpha: 0.1
  clf__norm: False
  tfidf__max_df: 0.5
  tfidf__min_df: 1
  tfidf__sublinear_tf: False


In [26]:
# Predecimos en test
best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test_docs)

f1_macro_test = f1_score(y_test, y_pred_test, average='macro')
f1_weighted_test = f1_score(y_test, y_pred_test, average='weighted')

print("=" * 60)
print(f"DESEMPEÑO EN CONJUNTO DE TEST (MEJOR MODELO):")
print(f"F1-Score (Macro):    {f1_macro_test:.4f}")
print(f"F1-Score (Weighted): {f1_weighted_test:.4f}")
print("=" * 60)

print("\nClassification Report en Test:")
print(classification_report(
    y_test, 
    y_pred_test, 
    target_names=newsgroups_train.target_names, 
    digits=4
))


DESEMPEÑO EN CONJUNTO DE TEST (MEJOR MODELO):
F1-Score (Macro):    0.6919
F1-Score (Weighted): 0.7050

Classification Report en Test:
                          precision    recall  f1-score   support

             alt.atheism     0.3231    0.4639    0.3810       319
           comp.graphics     0.7132    0.7095    0.7113       389
 comp.os.ms-windows.misc     0.7106    0.5609    0.6270       394
comp.sys.ibm.pc.hardware     0.6325    0.6760    0.6535       392
   comp.sys.mac.hardware     0.7584    0.7013    0.7287       385
          comp.windows.x     0.8000    0.7899    0.7949       395
            misc.forsale     0.7380    0.7077    0.7225       390
               rec.autos     0.7787    0.7374    0.7575       396
         rec.motorcycles     0.7984    0.7663    0.7821       398
      rec.sport.baseball     0.9076    0.8413    0.8732       397
        rec.sport.hockey     0.8552    0.9474    0.8989       399
               sci.crypt     0.7646    0.7955    0.7797       396
       

#### Analisis

Si comparamos el modelo obtenido con el baseline, vemos como este mejoro ampliamente ya que aumentamos el F1-Score Macro de 0.5 a 0.69. Ademas vemos que el modelo es muy bueno en ciertas categorias, regular en la mayoria y mala en una (`alt.atheism`). Es curioso ademas que nuevamente `rec.sport.hockey` sea la queu mejor resultado tiene, esto quiza sea por jerga muy propia de este tipo de noticias que siempre esta presente.

En este caso en particular, el mejor set de hiperparametros encontrados mediante la busqueda por grilla fueron:
- Complement Naive Bayes
    - alpha: 0.1
    - norm: False
- TF-IDF
    - max_df: 0.5
    - min_df: 1
    - sublinear_tf: False

### Cosnigna 4

In [65]:
#Transponemos X_train
X_train_t = X_train.transpose()

In [72]:
#Seleccionamos 5 palabras
words = ['crime', 'health', 'junior', 'break', 'rain']
for word in words:
    try:
        print(f'Indice correspondidente de {word} -> {tfidfvect.vocabulary_[word]}')
    except:
        print(f'Nunca se observo a {word}')

Indice correspondidente de crime -> 30490
Indice correspondidente de health -> 46076
Indice correspondidente de junior -> 52963
Indice correspondidente de break -> 24048
Indice correspondidente de rain -> 75921


In [73]:
feature_names = tfidfvect.get_feature_names_out()
for word in words:
  idx = tfidfvect.vocabulary_[word]
  cossim = cosine_similarity(X_train_t[idx], X_train_t)[0]
  mostsim = np.argsort(cossim)[::-1][1:6]
  print(f'Top 5 palabras mas similares de {word}:')
  for i in mostsim:
    similar_word = feature_names[i]
    print(f"  - {similar_word}")
  print('='*60)

Top 5 palabras mas similares de crime:
  - virutally
  - skyrocketed
  - nomn
  - eqaul
  - violent
Top 5 palabras mas similares de health:
  - ohip
  - provincial
  - care
  - breakoff
  - traditionalists
Top 5 palabras mas similares de junior:
  - salvage
  - fiance
  - bram
  - gorgeous
  - strapped
Top 5 palabras mas similares de break:
  - receipt
  - framing
  - uarts
  - 2661
  - 7201
Top 5 palabras mas similares de rain:
  - thatch
  - bj10ex
  - 500c
  - stealing
  - dries


### Analisis

* **crime (`virutally`, `skyrocketed`, `nomn`, `eqaul`, `violent`):**
  * Captura relaciones validas con `violent` y `skyrocketed`.
  * Aparecen errores tipográficos (`eqaul`, `virutally`). Seguramente son muy ruidoson y hayan aparecido una vez y casualmente en un documento que diga crime.

* **health (`ohip`, `provincial`, `care`, `breakoff`, `traditionalists`):**
  * Captura `ohip` que es el Ontario Health Insurance Plan y `care`. El resto de palabras no tienen mucho sentido.

* **junior (`salvage`, `fiance`, `bram`, `gorgeous`, `strapped`):**
  * En este caso es dificil trazar alguna relacion con la palabra, esto quiza se deba a que es una palabra que no tiene una pertenencia a un dominio especifico.

* **break (`receipt`, `framing`, `uarts`, `2661`, `7201`):**
  * Es un caso interesante ya que segun el corpus la palabra break tiene un significado semantico mas ligado a los protocolos de comunicacion seriales `uarts`, `2661` y `7201`. Esto muestra como esta codificacion sigue las tendencias del corpus con el que fue generado ya que uno asocia esta palabra a romper, no a las comunicaciones.

* **rain (`thatch`, `bj10ex`, `500c`, `stealing`, `dries`):**
  * Contiene términos asociados por sentido físico (`thatch` o techo de paja, `dries`). El resto carecen de sentido.

En general notamos que al trasponer la matriz se logra capturar cierta similud semantica. Ademas es intersente el caso de break como ejemplo de que el significado tiene un sesgo sobre el corpus usado, lo que peude dar resultados menos esperados
